In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np

In [22]:
from sklearn.preprocessing import LabelEncoder

In [23]:
data = pd.read_csv('pytorch_reg_cls/DateFruit_Dataset.csv')
data.head(5)

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,ROUNDNESS,COMPACTNESS,SHAPEFACTOR_1,SHAPEFACTOR_2,SHAPEFACTOR_3,SHAPEFACTOR_4,MeanRR,MeanRG,MeanRB,StdDevRR,StdDevRG,StdDevRB,SkewRR,SkewRG,SkewRB,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,0.9374,0.8750,0.0020,0.0015,0.7657,0.9936,117.4466,109.9085,95.6774,26.5152,23.0687,30.1230,-0.5661,-0.0114,0.6019,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,0.9773,0.9065,0.0021,0.0018,0.8218,0.9993,100.0578,105.6314,95.6610,27.2656,23.4952,28.1229,-0.2328,0.1349,0.4134,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,0.9446,0.8706,0.0018,0.0014,0.7580,0.9968,130.9558,118.5703,103.8750,29.7036,24.6216,33.9053,-0.7152,-0.1059,0.9183,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,0.9458,0.8791,0.0020,0.0016,0.7727,0.9915,86.7798,88.2531,82.3751,28.7288,24.4905,30.3955,0.4584,1.2917,1.8028,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,0.9358,0.8707,0.0022,0.0017,0.7582,0.9938,105.5484,101.8132,85.3342,30.3205,25.0291,27.1741,-0.3552,0.2101,0.8865,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [24]:
data.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [25]:
X = data.drop('Class', axis=1)
y = data['Class']

In [26]:
data.shape

(898, 35)

In [27]:
X.shape

(898, 34)

In [28]:
y.shape

(898,)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42, shuffle=True)

In [30]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [31]:
import optuna
from sklearn.model_selection import cross_val_score, KFold

In [32]:
def objective(trial):

    # set parameters space
    n_estimators = trial.suggest_int('n_estimators', 20, 250)
    criterion = trial.suggest_categorical('criterion', ['gini', 'log_loss', 'entropy'])
    max_depth = trial.suggest_int('max_depth', 3, 15)

    # define model
    model = RandomForestClassifier(
        n_estimators= n_estimators,
        criterion= criterion, 
        max_depth=max_depth,
        n_jobs=-1, 
        random_state=42
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    score = cross_val_score(model, X_train, y_train, cv = kf, scoring='accuracy').mean()

    return score

In [35]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-10-28 15:50:48,843] A new study created in memory with name: no-name-28050233-5691-4432-b211-13b168dcc4db
[I 2026-10-28 15:50:50,074] Trial 0 finished with value: 0.8663170163170163 and parameters: {'n_estimators': 68, 'criterion': 'gini', 'max_depth': 7}. Best is trial 0 with value: 0.8663170163170163.
[I 2026-10-28 15:50:51,444] Trial 1 finished with value: 0.878885003885004 and parameters: {'n_estimators': 184, 'criterion': 'log_loss', 'max_depth': 9}. Best is trial 1 with value: 0.878885003885004.
[I 2026-10-28 15:50:52,151] Trial 2 finished with value: 0.8676864801864802 and parameters: {'n_estimators': 120, 'criterion': 'gini', 'max_depth': 5}. Best is trial 1 with value: 0.878885003885004.
[I 2026-10-28 15:50:53,081] Trial 3 finished with value: 0.8788947163947164 and parameters: {'n_estimators': 147, 'criterion': 'entropy', 'max_depth': 15}. Best is trial 3 with value: 0.8788947163947164.
[I 2026-10-28 15:50:54,318] Trial 4 finished with value: 0.8816724941724943 and pa

In [37]:
study.best_value

0.8830808080808081

In [38]:
study.best_params

{'n_estimators': 202, 'criterion': 'entropy', 'max_depth': 10}

In [39]:
best_model = RandomForestClassifier(
    n_estimators=202,
    criterion='entropy',
    max_depth=10,
    n_jobs=-1, 
    random_state=42
)

best_model.fit(X_train, y_train)

print(f"Test Scorer: {best_model.score(X_test, y_test)}")

Test Scorer: 0.9333333333333333


In [ ]:
# visualizing optuna process

from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

In [41]:
plot_optimization_history(study)

In [43]:
# plot_intermediate_values(study)       # pruning feature to utilize `plot_intermediate_values()`

In [44]:
plot_parallel_coordinate(study)

In [45]:
plot_contour(study)

In [46]:
plot_slice(study)


In [47]:
plot_param_importances(study)
